# TF-IDF и BM25: расширение pool-разметки

Честное сравнение классических методов поиска (TF-IDF, BM25) с нашим
пайплайном требует, чтобы их top-K кандидаты тоже попали в pool и
получили экспертную оценку. Без этого любой пост, который нашли только
TF-IDF/BM25, автоматически засчитался бы как score = 0 — это стандартный
pooling bias.

Здесь:
1. Загружаем 50 000 постов из таблицы LanceDB `e5-base-fine-tuned-50k`.
2. Строим TF-IDF (sklearn) и BM25 (rank_bm25) индексы.
3. Для каждого из 17 запросов получаем top-20 от каждого метода.
4. Расширяем `benchmark/pipeline/pool_candidates.json` теми кандидатами,
   которых ещё нет в пуле (ключ — текст поста).
5. Сохраняем обновлённый пул в том же файле; existing-разметка в
   `ground_truth_pool.json` остаётся валидной (у новых кандидатов будут
   свежие `cand_idx`, старые — нетронуты).

После прогона этого ноутбука:
— откройте `benchmark/pipeline/pipeline-50k-major_metrics-annotation.ipynb`
— запустите ячейку разметки: она автоматически подхватит расширенный пул
  и продолжит с первой неразмеченной пары.


In [1]:
# Поднимаемся к корню thesis/, чтобы относительные пути работали
import os
from pathlib import Path
_p = Path.cwd()
if _p.name == 'baselines':
    os.chdir(_p.parent.parent)
print('CWD:', os.getcwd())


CWD: c:\Users\Admin\Documents\диплом\thesis


In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import time
import re
from collections import OrderedDict

import numpy as np
import pandas as pd

import lancedb


In [3]:
# ==================== КОНФИГУРАЦИЯ ====================

# Индекс — тот же, что и у финального пайплайна (честно: сравниваем на
# одной и той же базе из 50 000 постов)
LANCEDB_PATH      = './lancedb_store'
POOL_TABLE        = 'e5-base-fine-tuned-50k'

# Ground truth pool (расширяем тот же файл)
GT_PAIRS_JSON        = 'ground_truth_pairs.json'
POOL_CANDIDATES_JSON = 'benchmark/pipeline/pool_candidates.json'

# Дубли (как в pipeline-50k-major_metrics-annotation.ipynb)
DROP_QUERY_INDEXES = {6, 14}

# Сколько постов брать от каждого метода в pool
TOP_K = 20

# Пороги русскоязычной токенизации
TOKEN_RE = re.compile(r'[А-Яа-яЁёA-Za-z0-9]+')


## 1. Запросы (те же 17)

In [4]:
with open(GT_PAIRS_JSON, encoding='utf-8') as f:
    all_pairs = json.load(f)

queries = []
for i, pair in enumerate(all_pairs, start=1):
    if i in DROP_QUERY_INDEXES:
        continue
    queries.append({
        'query_idx':   i,
        'imt_name':    pair['imt_name'],
        'description': pair['description'],
    })
print(f'Запросов: {len(queries)} (пропущены дубли: {sorted(DROP_QUERY_INDEXES)})')


Запросов: 17 (пропущены дубли: [6, 14])


## 2. Загрузка 50k постов

In [5]:
db = lancedb.connect(LANCEDB_PATH)
tbl = db.open_table(POOL_TABLE)
# LanceDB: читаем все строки, нам нужен только текст и метаданные
df = tbl.to_pandas()
print(f'Постов в таблице {POOL_TABLE}: {len(df):,}')
print('Колонки:', list(df.columns))
# В таблице эмбеддинги нам не нужны — оставляем только текст
posts_df = df[['text', 'channel', 'category']].reset_index(drop=True)
print(posts_df.head(2))


Постов в таблице e5-base-fine-tuned-50k: 50,000
Колонки: ['vector', 'text', 'channel', 'category', 'post_id', 'link', 'date', 'views']
                                                text        channel  \
0  Ма Тут активация пишет недействительны, значит...    obshakstaya   
1  Привет, девочки, нужна ваша помощь, обращаюсь ...  moda_ais_sale   

         category  
0           Блоги  
1  Мода и красота  


## 3. TF-IDF индекс

Классическая реализация: `sklearn.feature_extraction.text.TfidfVectorizer`
на словах (1-граммы + 2-граммы для устойчивости к флексии), l2-нормировка.
Запрос векторизуется тем же vectorizer'ом, сходство — косинусное.


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel


def simple_tokenizer(text):
    return TOKEN_RE.findall(text.lower())


t0 = time.time()
tfidf_vec = TfidfVectorizer(
    tokenizer=simple_tokenizer,
    lowercase=False,          # уже lowercase в токенизаторе
    ngram_range=(1, 2),
    min_df=2,                 # отсекаем крайне редкие термины
    max_df=0.95,              # отсекаем стоп-подобные
    norm='l2',
)
tfidf_matrix = tfidf_vec.fit_transform(posts_df['text'].tolist())
print(f'TF-IDF матрица: {tfidf_matrix.shape}, построена за {time.time()-t0:.1f}с')


TF-IDF матрица: (50000, 493042), построена за 5.3с


In [7]:
def tfidf_top_k(query_text, k=TOP_K):
    q_vec = tfidf_vec.transform([query_text])
    sims = linear_kernel(q_vec, tfidf_matrix).flatten()  # cos sim, т.к. оба l2-normalized
    top_idx = np.argpartition(-sims, kth=k)[:k]
    # сортируем окончательно
    top_idx = top_idx[np.argsort(-sims[top_idx])]
    return [(int(i), float(sims[i])) for i in top_idx]


# Быстрый sanity-check на первом запросе
_sample = tfidf_top_k(queries[0]['description'], 3)
print(f'TF-IDF top-3 для «{queries[0]["imt_name"]}»:')
for idx, sc in _sample:
    print(f'  score={sc:.3f} | {posts_df.iloc[idx]["text"][:100]!r}')


TF-IDF top-3 для «Затирка для плитки готовая - белая»:
  score=0.157 | 'СУПЕР СРЕДСТВО УДАЛЯЕТ ЧЕРНУЮ ПЛЕСЕНЬ и ГРИБОК за 30 минут Фунгицид в составе обладает дезинфицирующ'
  score=0.141 | 'Процесс замены старого герметика в ванной.1. Срезаем старый герметик.2. Убираем все остатки канцеляр'
  score=0.110 | 'Составы команд Сидакова и Садулаева на матчевую встречу 19 августа.Команда победителей получит 500 т'


## 4. BM25 индекс

`rank_bm25.BM25Okapi` на той же токенизации.


In [8]:
# pip install rank_bm25 должно быть установлено в окружении thesis.
# Если нет — раскомментировать:
# !pip install rank_bm25 --quiet

from rank_bm25 import BM25Okapi

t0 = time.time()
tokenized_corpus = [simple_tokenizer(t) for t in posts_df['text'].tolist()]
bm25 = BM25Okapi(tokenized_corpus)
print(f'BM25 индекс построен за {time.time()-t0:.1f}с')


BM25 индекс построен за 1.5с


In [9]:
def bm25_top_k(query_text, k=TOP_K):
    q_tokens = simple_tokenizer(query_text)
    scores = bm25.get_scores(q_tokens)
    top_idx = np.argpartition(-scores, kth=k)[:k]
    top_idx = top_idx[np.argsort(-scores[top_idx])]
    return [(int(i), float(scores[i])) for i in top_idx]


_sample = bm25_top_k(queries[0]['description'], 3)
print(f'BM25 top-3 для «{queries[0]["imt_name"]}»:')
for idx, sc in _sample:
    print(f'  score={sc:.3f} | {posts_df.iloc[idx]["text"][:100]!r}')


BM25 top-3 для «Затирка для плитки готовая - белая»:
  score=88.351 | 'СУПЕР СРЕДСТВО УДАЛЯЕТ ЧЕРНУЮ ПЛЕСЕНЬ и ГРИБОК за 30 минут Фунгицид в составе обладает дезинфицирующ'
  score=71.901 | 'Нижнее белье трусы и топ Комплект нижнего белья для женщин подходит как для повседневной жизни, так '
  score=66.174 | 'Ультрафиолетовая сушилка для обуви#Техника Цена: 1 703₽ 14 900₽ (Скидка -89%)В сезон осенних дождей '


## 5. Расширение pool

Для каждого запроса:
1. Берём TF-IDF top-20 и BM25 top-20.
2. Смотрим, какие из этих постов уже есть в `pool_candidates.json`.
3. Новые добавляем как новые кандидаты (с продолжением нумерации `cand_idx`).
4. Если пост уже был — **не дублируем**, а только добавляем соответствующий
   флаг в `source`, чтобы в метриках было понятно, что метод его нашёл.

Структура записи в пуле расширяется: помимо `source ∈ {bi, rerank}`
появляются значения `tfidf` и `bm25`. Существующие аннотации в
`ground_truth_pool.json` остаются корректными (они привязаны к
`query_idx:cand_idx` — а эти ключи не меняются).


In [10]:
# Для каждого метода сохраняем, какие тексты он вытащил в top-K — нужны потом
# для вычисления метрик и для отметки source.
method_hits = {
    'tfidf': {},  # query_idx -> {text: rank}
    'bm25':  {},
}
for q in queries:
    for meth, fn in [('tfidf', tfidf_top_k), ('bm25', bm25_top_k)]:
        top = fn(q['description'], TOP_K)
        method_hits[meth][q['query_idx']] = OrderedDict(
            (posts_df.iloc[idx]['text'], rank) for rank, (idx, _sc) in enumerate(top, start=1)
        )
print('Готово: кандидаты собраны для обоих методов')
print(f'  TF-IDF: {sum(len(v) for v in method_hits["tfidf"].values())} пар (с учётом дублей)')
print(f'  BM25:   {sum(len(v) for v in method_hits["bm25"].values())} пар (с учётом дублей)')


Готово: кандидаты собраны для обоих методов
  TF-IDF: 295 пар (с учётом дублей)
  BM25:   245 пар (с учётом дублей)


In [11]:
# Загружаем существующий пул
with open(POOL_CANDIDATES_JSON, encoding='utf-8') as f:
    pool = json.load(f)

# Индексируем по query_idx
pool_by_q = {p['query_idx']: p for p in pool}

new_pairs_total = 0
updated_sources_total = 0

for q in queries:
    qi = q['query_idx']
    p = pool_by_q[qi]
    existing_texts = {c['text'].strip(): ci for ci, c in enumerate(p['candidates'])}

    for meth in ('tfidf', 'bm25'):
        for text in method_hits[meth][qi]:
            key = text.strip()
            if key in existing_texts:
                # обновить source — помечаем, что метод нашёл этот пост
                ci = existing_texts[key]
                src = p['candidates'][ci].setdefault('source', [])
                if meth not in src:
                    src.append(meth)
                    updated_sources_total += 1
            else:
                # добавить нового кандидата — строка поста в таблице постов,
                # метаданные подтягиваем по тексту (уникальному)
                row = posts_df[posts_df['text'] == text].iloc[0]
                new_cand = {
                    'text':     text,
                    'channel':  row['channel'],
                    'category': row.get('category', ''),
                    'source':   [meth],
                }
                p['candidates'].append(new_cand)
                existing_texts[key] = len(p['candidates']) - 1
                new_pairs_total += 1

print(f'Добавлено новых кандидатов (пар для разметки): {new_pairs_total}')
print(f'Обновлены source-метки у существующих кандидатов: {updated_sources_total}')
print(f'Итоговый размер пула: {sum(len(p["candidates"]) for p in pool)} пар по {len(pool)} запросам')


Добавлено новых кандидатов (пар для разметки): 368
Обновлены source-метки у существующих кандидатов: 172
Итоговый размер пула: 814 пар по 17 запросам


In [12]:
# Сохраняем
with open(POOL_CANDIDATES_JSON, 'w', encoding='utf-8') as f:
    json.dump(pool, f, ensure_ascii=False, indent=2)
print(f'Сохранено в {POOL_CANDIDATES_JSON}')


Сохранено в benchmark/pipeline/pool_candidates.json


## 6. Что делать дальше

1. Открыть `benchmark/pipeline/pipeline-50k-major_metrics-annotation.ipynb`.
2. Прогнать ячейки сверху вниз — ячейка разметки автоматически покажет:
   «Уже размечено: 446 / {новый размер}». Можно разметить оставшиеся.
3. После окончания разметки — прогнать
   `benchmark/baselines/tfidf-bm25-evaluate.ipynb` (он посчитает
   метрики всех 4 методов на одном и том же финальном пуле).


In [13]:
# Итоговый отчёт по запросам: сколько новых пар у какого запроса
stats = []
for p in pool:
    n_total = len(p['candidates'])
    n_new = sum(1 for c in p['candidates'] if set(c.get('source', [])) <= {'tfidf', 'bm25'})
    stats.append({
        'query_idx': p['query_idx'],
        'imt_name':  p['imt_name'][:50],
        'total':     n_total,
        'new':       n_new,
    })
pd.DataFrame(stats)


,query_idx,imt_name,total,new
0,1,Затирка для плитки готовая - белая,55,29
1,2,Самоклеящиеся панели для стен на кухню 60х30см...,41,18
2,3,Развивашки 2-3-4 года/пиши стирай тетрадь/книг...,47,19
3,4,"Детская мозаика (5 цветов, 40 элементов) ""Кора...",58,24
4,5,Жиросжигатель для похудения женщинам 60 капсул,52,24
5,7,Накидка на сиденье DongFeng Fengshen Yixuan GS,50,26
6,8,Кроссовер Monjaro,44,19
7,9,Матрас надувной двуспальный 203х152см с подушк...,47,23
8,10,Гуд Найт Мягкое фито снотворное для сна,49,22
9,11,Клавиатура игровая с подсветкой\n,42,15
